# UN GA Voting Data: CSV → Agreement Matrix
Transforms the raw per-country-per-resolution vote CSV into a pairwise country agreement score CSV and JSON.

**Input:** `local-data/2026_02_06_ga_voting.csv`  
**Outputs:** `un_voting_agreement.csv`, `un_voting_data.json`

## 1. Load & Inspect

In [6]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv('local-data/2026_02_06_ga_voting.csv', low_memory=False)

print('Total rows:', len(df))
print('Columns:', df.columns.tolist())
print('Vote values:', df['ms_vote'].unique())
print('Date range:', df['date'].min(), '-', df['date'].max())
print('Countries:', df['ms_code'].nunique())
print('Resolutions:', df['resolution'].nunique())

Total rows: 947434
Columns: ['undl_id', 'ms_code', 'ms_name', 'ms_vote', 'date', 'session', 'resolution', 'draft', 'committee_report', 'meeting', 'title', 'agenda_title', 'subjects', 'vote_note', 'total_yes', 'total_no', 'total_abstentions', 'total_non_voting', 'total_ms', 'undl_link']
Vote values: ['Y' 'X' 'A' 'N']
Date range: 1946-01-26 - 2025-12-18
Countries: 202
Resolutions: 5694


## 2. Feature Engineering: Year, Decade, Primary Subject

In [7]:
# Parse year and derive decade
df['year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
df['decade'] = (df['year'] // 10 * 10).fillna(0).astype(int)

# Subjects are pipe-separated; take the first (broadest) topic
def primary_subject(s):
    if pd.isna(s):
        return 'UNCLASSIFIED'
    return s.split('|')[0].strip()

df['primary_subject'] = df['subjects'].apply(primary_subject)

print('Decades:', sorted(df['decade'].unique()))
print('Unique primary subjects:', df['primary_subject'].nunique())

Decades: [1940, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]
Unique primary subjects: 398


## 3. Filter to Substantive Votes Only

Vote codes: `Y` = Yes, `N` = No, `A` = Abstain, `X` = Non-voting/Absent  
We drop `X` — only positions that represent a deliberate stance are counted.

In [8]:
votes = df[df['ms_vote'].isin(['Y', 'N', 'A'])].copy()
print('Rows after filtering:', len(votes))

Rows after filtering: 859357


## 4. Core Function: Compute Pairwise Agreement Scores

For every pair of countries in a given subset of votes:
- Find all resolutions where **both** countries cast a vote.
- Score = (# resolutions where they voted the same) / (# resolutions where both voted).
- Skip pairs with fewer than 5 co-votes (too little data).

In [9]:
def compute_agreement_pairs(subset, min_shared=5):
    """
    Returns a list of (country_a, country_b, score, pct_yy, pct_nn, pct_aa) tuples.
    - score   : fraction of shared votes where both voted the same
    - pct_yy  : fraction of shared votes where both voted Yes
    - pct_nn  : fraction of shared votes where both voted No
    - pct_aa  : fraction of shared votes where both voted Abstain
    Only upper-triangle pairs are returned (a < b lexicographically).
    """
    pivot = subset.pivot_table(
        index='resolution', columns='ms_code', values='ms_vote', aggfunc='first'
    )
    countries = pivot.columns.tolist()
    arr = pivot.values  # shape: (resolutions, countries)

    results = []
    for i in range(len(countries)):
        for j in range(i + 1, len(countries)):
            c1, c2 = countries[i], countries[j]
            v1, v2 = arr[:, i], arr[:, j]
            # Both countries must have voted on the same resolution
            mask = (~pd.isna(v1)) & (~pd.isna(v2))
            total = mask.sum()
            if total < min_shared:
                continue
            v1m, v2m = v1[mask], v2[mask]
            agree  = (v1m == v2m).sum()
            pct_yy = round(int(((v1m == 'Y') & (v2m == 'Y')).sum()) / total, 4)
            pct_nn = round(int(((v1m == 'N') & (v2m == 'N')).sum()) / total, 4)
            pct_aa = round(int(((v1m == 'A') & (v2m == 'A')).sum()) / total, 4)
            results.append((c1, c2, round(agree / total, 4), pct_yy, pct_nn, pct_aa))
    return results

## 5. Overall Agreement (All Time, All Subjects)

In [10]:
print('Computing overall agreement matrix...')
overall_pairs = compute_agreement_pairs(votes)

overall_rows = [
    {'country_a': c1, 'country_b': c2, 'score': score,
     'pct_yy': pct_yy, 'pct_nn': pct_nn, 'pct_aa': pct_aa,
     'decade': 'all', 'subject': 'all'}
    for c1, c2, score, pct_yy, pct_nn, pct_aa in overall_pairs
]

print(f'Overall pairs: {len(overall_rows)}')

Computing overall agreement matrix...
Overall pairs: 19829


## 6. By-Decade Agreement

In [11]:
decade_rows = []

for dec in sorted(votes['decade'].unique()):
    sub = votes[votes['decade'] == dec]
    if sub['resolution'].nunique() < 5:
        continue
    pairs = compute_agreement_pairs(sub)
    for c1, c2, score, pct_yy, pct_nn, pct_aa in pairs:
        decade_rows.append({
            'country_a': c1, 'country_b': c2,
            'score': score, 'pct_yy': pct_yy, 'pct_nn': pct_nn, 'pct_aa': pct_aa,
            'decade': str(dec), 'subject': 'all'
        })
    print(f'  {dec}s: {len(pairs)} pairs')

print(f'Total decade rows: {len(decade_rows)}')

  1940s: 1711 pairs
  1950s: 3402 pairs
  1960s: 7984 pairs
  1970s: 11159 pairs
  1980s: 12403 pairs
  1990s: 17297 pairs
  2000s: 18710 pairs
  2010s: 18528 pairs
  2020s: 18528 pairs
Total decade rows: 109722


## 7. By-Subject Agreement (Top 25 Subjects)

Select the 25 subjects with the most resolutions, excluding `UNCLASSIFIED`, using a minimum of 3 shared votes per pair.

In [12]:
top_subjects = (
    votes.groupby('primary_subject')['resolution']
    .nunique()
    .sort_values(ascending=False)
    .drop('UNCLASSIFIED', errors='ignore')
    .head(25)
    .index.tolist()
)

print('Top subjects selected:')
for s in top_subjects:
    print(' -', s)

Top subjects selected:
 - DISARMAMENT--GENERAL AND COMPLETE
 - HUMAN RIGHTS ADVANCEMENT
 - TERRITORIES OCCUPIED BY ISRAEL--HUMAN RIGHTS--REPORTS
 - UNRWA--ACTIVITIES
 - PALESTINE QUESTION
 - HUMAN RIGHTS--REPORTS
 - DECOLONIZATION
 - MIDDLE EAST SITUATION
 - SELF-DETERMINATION OF PEOPLES
 - APARTHEID
 - UN. ECONOMIC AND SOCIAL COUNCIL--REPORTS
 - DISARMAMENT--UN. GENERAL ASSEMBLY (10TH SPECIAL SESS. : 1978)--RECOMMENDATIONS
 - INTERNATIONAL TRADE
 - LAW OF THE SEA
 - NON-SELF-GOVERNING TERRITORIES--REPORTS
 - DECOLONIZATION--UN SYSTEM
 - UN INTERIM FORCE IN LEBANON--FINANCING
 - SUSTAINABLE DEVELOPMENT
 - DISARMAMENT--UN. GENERAL ASSEMBLY (12TH SPECIAL SESS. : 1982)--RECOMMENDATIONS
 - NON-NUCLEAR-WEAPON STATES--SECURITY
 - NUCLEAR WEAPON TESTS--TREATIES
 - ARMS RACE--OUTER SPACE
 - CUBA--UNITED STATES
 - NAMIBIA QUESTION
 - NUCLEAR DISARMAMENT


In [13]:
subject_rows = []

for subj in top_subjects:
    sub = votes[votes['primary_subject'] == subj]
    pairs = compute_agreement_pairs(sub, min_shared=3)
    for c1, c2, score, pct_yy, pct_nn, pct_aa in pairs:
        subject_rows.append({
            'country_a': c1, 'country_b': c2,
            'score': score, 'pct_yy': pct_yy, 'pct_nn': pct_nn, 'pct_aa': pct_aa,
            'decade': 'all', 'subject': subj
        })
    print(f'  {subj}: {len(pairs)} pairs')

print(f'Total subject rows: {len(subject_rows)}')

  DISARMAMENT--GENERAL AND COMPLETE: 19702 pairs
  HUMAN RIGHTS ADVANCEMENT: 18900 pairs
  TERRITORIES OCCUPIED BY ISRAEL--HUMAN RIGHTS--REPORTS: 19695 pairs
  UNRWA--ACTIVITIES: 19697 pairs
  PALESTINE QUESTION: 19675 pairs
  HUMAN RIGHTS--REPORTS: 18889 pairs
  DECOLONIZATION: 19670 pairs
  MIDDLE EAST SITUATION: 19643 pairs
  SELF-DETERMINATION OF PEOPLES: 19682 pairs
  APARTHEID: 15185 pairs
  UN. ECONOMIC AND SOCIAL COUNCIL--REPORTS: 16577 pairs
  DISARMAMENT--UN. GENERAL ASSEMBLY (10TH SPECIAL SESS. : 1978)--RECOMMENDATIONS: 15115 pairs
  INTERNATIONAL TRADE: 19602 pairs
  LAW OF THE SEA: 19211 pairs
  NON-SELF-GOVERNING TERRITORIES--REPORTS: 19631 pairs
  DECOLONIZATION--UN SYSTEM: 19628 pairs
  UN INTERIM FORCE IN LEBANON--FINANCING: 17917 pairs
  SUSTAINABLE DEVELOPMENT: 18521 pairs
  DISARMAMENT--UN. GENERAL ASSEMBLY (12TH SPECIAL SESS. : 1982)--RECOMMENDATIONS: 17232 pairs
  NON-NUCLEAR-WEAPON STATES--SECURITY: 19125 pairs
  NUCLEAR WEAPON TESTS--TREATIES: 19616 pairs
  ARMS

## 8. Combine & Save CSV

In [14]:
combined = pd.concat([
    pd.DataFrame(overall_rows),
    pd.DataFrame(decade_rows),
    pd.DataFrame(subject_rows)
], ignore_index=True)

print(f'Combined rows: {len(combined)}')
print(combined.head())

combined.to_csv('un_voting_agreement.csv', index=False)
print('Saved: un_voting_agreement.csv')

Combined rows: 591116
  country_a country_b   score  pct_yy  pct_nn  pct_aa decade subject
0       AFG       AGO  0.9333  0.9055  0.0089  0.0189    all     all
1       AFG       ALB  0.7026  0.6875  0.0108  0.0043    all     all
2       AFG       AND  0.6160  0.6129  0.0000  0.0031    all     all
3       AFG       ARE  0.9202  0.9081  0.0024  0.0097    all     all
4       AFG       ARG  0.8013  0.7860  0.0006  0.0146    all     all
Saved: un_voting_agreement.csv


## 9. Build JSON Lookup & Save

Structure: `{ meta: {countries, decades, subjects}, data: { "decade|subject": [[cA, cB, score], ...] } }`

In [15]:
# Metadata
all_countries = sorted(set(combined['country_a'].tolist() + combined['country_b'].tolist()))
decades = ['all'] + sorted(
    [str(d) for d in combined['decade'].unique() if d != 'all'],
    key=lambda x: int(x)
)
subjects = sorted(combined['subject'].unique())

meta = {
    'countries': all_countries,
    'decades': decades,
    'subjects': subjects
}

# Per-country voting stats: [pct_yes, pct_no, pct_abstain, pct_absent]
# Uses the full df (including X/absent) so all four fractions sum to 1.
country_stats = {}
for country, grp in df.groupby('ms_code'):
    total = len(grp)
    if total == 0:
        continue
    y = round(int((grp['ms_vote'] == 'Y').sum()) / total, 3)
    n = round(int((grp['ms_vote'] == 'N').sum()) / total, 3)
    a = round(int((grp['ms_vote'] == 'A').sum()) / total, 3)
    x = round(max(0.0, 1.0 - y - n - a), 3)
    country_stats[country] = [y, n, a, x]

print(f'Country stats computed for {len(country_stats)} countries')

# Data: flat arrays per combo key
# Each entry: [cA, cB, score, pct_yy, pct_nn, pct_aa]
#   score   = overall agreement fraction
#   pct_yy  = fraction of shared votes both voted Yes
#   pct_nn  = fraction of shared votes both voted No
#   pct_aa  = fraction of shared votes both voted Abstain
#   implied: pct_disagree = 1 - score
data = {}
for (decade, subject), group in combined.groupby(['decade', 'subject']):
    key = f'{decade}|{subject}'
    data[key] = [
        [row['country_a'], row['country_b'],
         round(row['score'], 3),
         round(row['pct_yy'], 3),
         round(row['pct_nn'], 3),
         round(row['pct_aa'], 3)]
        for _, row in group.iterrows()
    ]

bundle = {'meta': meta, 'data': data, 'stats': country_stats}

with open('un_voting_data.json', 'w') as f:
    json.dump(bundle, f, separators=(',', ':'))

import os
print(f'Saved: un_voting_data.json ({os.path.getsize("un_voting_data.json")/1024/1024:.1f} MB)')

Country stats computed for 202 countries
Saved: un_voting_data.json (19.1 MB)
